In [1]:
import json
from pathlib import Path
import google.generativeai as genai
import os
from langchain_core.prompts import PromptTemplate
import time
from dotenv import load_dotenv

/home/haseebmuhammad/miniconda3/envs/rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gemini_api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key = gemini_api_key)
model = genai.GenerativeModel('gemini-pro')

prompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that generates QA pairs from a given text extracted from a book. Generate specific question-and-answer pairs from the given input text.

    Input Text:
    {input}

    Instruction: Return a JSON object in the response that strictly matches the following structure:

    [
        {{
            "Question": "Example question 1",
            "Answer": "Example answer 1"
        }},
        {{
            "Question": "Example question 2",
            "Answer": "Example answer 2"
        }}
        // ... additional question-answer pairs ...
    ]

    Output:
""")

adaptorPrompt = PromptTemplate.from_template("""
    Task: You are an AI assistant that converts given text into correct json format.

                                                
    Input Text: 
    {input}
                                                                                                                              
    Instruction: Return a JSON object in the response that strictly matches the following structure:

        [
            {{
                "Question": "Example question 1",
                "Answer": "Example answer 1"
            }},
            {{
                "Question": "Example question 2",
                "Answer": "Example answer 2"
            }}
            // ... additional question-answer pairs ...
        ]
""")


In [3]:
def generate_question_answer(chunk):
    formatted_prompt = prompt.format(input=chunk['input_text'])
    # print(f"{formatted_prompt}=")

    try:
        response = model.generate_content(formatted_prompt).text
        print(f"{response=}")
        formatted_adapted_prompt = adaptorPrompt.format(input=response)
        chunk['qa_pairs'] = json.loads(response)
    except json.JSONDecodeError:
            while True:
                try:
                    time.sleep(5)
                    response = model.generate_content(formatted_prompt).text
                    print(f"{response=}")
                    chunk['qa_pairs'] = json.loads(response)
                    break
                except json.JSONDecodeError:
                    continue
    return response 
    

In [4]:
chunks_path = Path('booksChunks')
dataset_path = Path('Dataset')

for file in chunks_path.glob("*.json"):
    with open(file, 'r') as json_file:
        chunks_dict = json.load(json_file)
    count=0
    for chunk in chunks_dict:
        # Generate QA pairs and parse them as JSON
        chunk = generate_question_answer(chunk)
        print(chunk)
        print(f"Chunk {count} done")
        count+=1
        # if count==5:
        #     break
        
    # Save the updated chunks back to the file
    with open(os.path.join(dataset_path, os.path.basename(file)), 'w') as json_file:
        json.dump(chunks_dict, json_file, indent=2)


response='```json\n[]\n```'
response='[\n  {\n    "Question": "Who are the authors of the book \\"Fundamentals of Deep Learning\\"?",\n    "Answer": "Nithin Buduma, Nikhil Buduma, and Joe Papa"\n  },\n  {\n    "Question": "Who contributed to the book \\"Fundamentals of Deep Learning\\"?",\n    "Answer": "Nicholas Locascio"\n  }\n]'
[
  {
    "Question": "Who are the authors of the book \"Fundamentals of Deep Learning\"?",
    "Answer": "Nithin Buduma, Nikhil Buduma, and Joe Papa"
  },
  {
    "Question": "Who contributed to the book \"Fundamentals of Deep Learning\"?",
    "Answer": "Nicholas Locascio"
  }
]
Chunk 0 done
response='[\n  {\n    "Question": "Where can you find more information about the book?",\n    "Answer": "http://oreilly.com"\n  },\n  {\n    "Question": "Who are the editors of the book?",\n    "Answer": "Melissa Potter and Rebecca Novack"\n  },\n  {\n    "Question": "Who is the production editor for the book?",\n    "Answer": "Katherine Tozer"\n  },\n  {\n    "Questio

KeyboardInterrupt: 